# Meshing!
For this project we need a structure to hold the mesh used for the calculations.

I'll create a dict that holds these keys:
- "dim" - dimensionality of the mesh (2 or 3)
- "points" - array of tuples for coordinates in of points in the mesh
- "faces" - array of tupes with indecies of the points in each face
- "edges" - array of tuples with indecies of the points in each edge

In [2]:
import numpy as np
import plotly.graph_objects as go

In [23]:
def grid(Nx, Ny):
    points = [(x, y) for y in range(Ny) for x in range(Nx)]
    
    def p_ind(x,y):
        return y*Nx + x
    
    faces = []
    for i in range(Nx - 1):
        for j in range(Ny - 1):
            faces.extend( [(p_ind(i,j), p_ind(i+1, j), p_ind(i+1, j+1)), (p_ind(i,j), p_ind(i, j+1), p_ind(i+1, j+1))])
    
    edges = set()
    for face in faces:

            edges.add( (min(face[0], face[1]), max(face[0], face[1])) )
            edges.add( (min(face[1], face[2]), max(face[1], face[2])) )
            edges.add( (min(face[0], face[2]), max(face[0], face[2])) )
    edges = list(edges)
    
    return dict(
        dim = 2,
        points = points,
        faces = faces,
        edges = edges
    )

      
data = grid(2,3)

In [25]:
import numpy as np
from scipy.spatial import Voronoi, Delaunay

def centroid_of_polygon(poly):
    """Return centroid of a polygon given as Nx2 array."""
    x = poly[:, 0]
    y = poly[:, 1]
    a = np.sum(x[:-1] * y[1:] - x[1:] * y[:-1]) / 2.0
    if abs(a) < 1e-14:
        return poly.mean(axis=0)
    cx = np.sum((x[:-1] + x[1:]) * (x[:-1] * y[1:] - x[1:] * y[:-1])) / (6 * a)
    cy = np.sum((y[:-1] + y[1:]) * (x[:-1] * y[1:] - x[1:] * y[:-1])) / (6 * a)
    return np.array([cx, cy])


def clip_polygon_circle(poly, R):
    """Clip polygon to the circle (simple radial clipping)."""
    # Project outside vertices back to the circle
    clipped = []
    for x, y in poly:
        r = np.sqrt(x*x + y*y)
        if r > R:
            x = x * R / r
            y = y * R / r
        clipped.append([x, y])
    return np.array(clipped)


def circle_cvt(N_interior=300, N_boundary=80, R=1.0, n_iter=5):
    """
    Perform CVT (Lloyd relaxation) inside a circle.
    Boundary points remain fixed on the circle.
    Interior points move to centroids of clipped Voronoi cells.
    """
    # ----- initial boundary points -----
    theta_b = np.linspace(0, 2*np.pi, N_boundary, endpoint=False)
    boundary = np.stack([R * np.cos(theta_b), R * np.sin(theta_b)], axis=1)

    # ----- initial interior points -----
    u = np.random.rand(N_interior)
    v = np.random.rand(N_interior)
    r = R * np.sqrt(u)
    theta = 2*np.pi * v
    interior = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)

    for _ in range(n_iter):
        pts = np.vstack([interior, boundary])
        vor = Voronoi(pts)

        new_interior = []

        for i in range(N_interior):  # interior only
            region_index = vor.point_region[i]
            region = vor.regions[region_index]
            if -1 in region or len(region) == 0:
                # Unbounded region: keep point as-is
                new_interior.append(interior[i])
                continue

            polygon = np.array([vor.vertices[v] for v in region])
            polygon = clip_polygon_circle(polygon, R)
            centroid = centroid_of_polygon(np.vstack([polygon, polygon[0]]))
            new_interior.append(centroid)

        interior = np.array(new_interior)

    # ----- final Delaunay triangulation -----
    final_points = np.vstack([interior, boundary])
    tri = Delaunay(final_points)

    # gather faces inside the circle
    faces = []
    for a, b, c in tri.simplices:
        pa, pb, pc = final_points[a], final_points[b], final_points[c]
        if (
            np.dot(pa, pa) <= R*R*1.00001 and
            np.dot(pb, pb) <= R*R*1.00001 and
            np.dot(pc, pc) <= R*R*1.00001
        ):
            faces.append((a, b, c))

    # unique edges
    edges = set()
    for a, b, c in faces:
        edges.add(tuple(sorted((a, b))))
        edges.add(tuple(sorted((b, c))))
        edges.add(tuple(sorted((a, c))))
    edges = list(edges)

    pts_list = [tuple(p) for p in final_points]

    return dict(
        dim=2,
        points=pts_list,
        faces=faces,
        edges=edges
    )
data = circle_cvt(
    N_interior=400,
    N_boundary=100,
    R=1.0,
    n_iter=8      # more iterations = smoother mesh
)


In [26]:
import plotly.graph_objects as go

def plot_mesh_plotly(data):
    points = data["points"]
    faces = data["faces"]

    xs = [p[0] for p in points]
    ys = [p[1] for p in points]

    # build edges from faces
    edge_x = []
    edge_y = []

    for f in faces:
        for (a, b) in [(0,1), (1,2), (2,0)]:
            x0, y0 = points[f[a]]
            x1, y1 = points[f[b]]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

    fig = go.Figure()

    # edges
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y,
        mode="lines",
        line=dict(width=2),
        hoverinfo="none"
    ))

    # points
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="markers",
        marker=dict(size=6),
        hoverinfo="text",
        text=[str(i) for i in range(len(points))]
    ))

    fig.update_layout(
        title="Grid Mesh",
        width=650,
        height=650,
        xaxis=dict(scaleanchor="y"),
        yaxis=dict(),
        showlegend=False
    )

    fig.show()

# example
# data = grid(6,20)
plot_mesh_plotly(data)


In [33]:
import numpy as np

def mobius_from_grid(Nx, Ny, width=0.2):
    """
    Take your 2D grid mesh and map each point onto a 3D Möbius strip.
    Nx, Ny are the same as in your grid() function.
    width is half-width of the strip in parameter v.
    """

    # --- get 2D grid mesh ---
    base = grid(Nx, Ny)
    points2d = base["points"]   # list of (x, y)

    # Convert (x, y) indices into parameters u, v
    # u ∈ [0, 2π], v ∈ [-width, width]
    points3d = []

    for (i, j) in points2d:
        u = 2 * np.pi * (i / (Nx))     # 0 → 2π
        v = width * (2 * (j / (Ny - 1)) - 1)  # 0→-w, 1→+w

        # Möbius mapping:
        X = (1 + (v/2) * np.cos(u/2)) * np.cos(u)
        Y = (1 + (v/2) * np.cos(u/2)) * np.sin(u)
        Z = (v/2) * np.sin(u/2)

        points3d.append((X, Y, Z))

    # faces + edges remain unchanged
    return dict(
        dim=3,
        points=points3d,
        faces=base["faces"],
        edges=base["edges"]
    )

data = mobius_from_grid(20, 4, width=.2)

In [34]:
import plotly.graph_objects as go

def plot_mesh_3D(data):
    points = data["points"]
    faces = data["faces"]

    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    zs = [p[2] for p in points]

    # build edges from faces
    edge_x = []
    edge_y = []
    edge_z = []

    for f in faces:
        for (a, b) in [(0,1), (1,2), (2,0)]:
            x0, y0, z0 = points[f[a]]
            x1, y1, z1 = points[f[b]]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]

    fig = go.Figure()

    # edges
    fig.add_trace(go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode="lines",
        line=dict(width=2),
        hoverinfo="none"
    ))

    # points
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys,z=zs,
        mode="markers",
        marker=dict(size=6),
        hoverinfo="text",
        text=[str(i) for i in range(len(points))]
    ))

    fig.update_layout(
        title="Grid Mesh",
        width=650,
        height=650,
        xaxis=dict(scaleanchor="y"),
        yaxis=dict(),
        showlegend=False
    )

    fig.show()

# example
# data = grid(6,20)
plot_mesh_3D(data)
